This notebook is used to creat sythesis pipeline fortop 20 recomended molecules using BioPKS

In [1]:
import sys
import os

# Completely suppress stderr output
sys.stderr = open(os.devnull, 'w')

# Now import everything
import warnings
warnings.filterwarnings('ignore')

In [6]:
import numpy as np
import pandas as pd
from biopks_pipeline import biopks_pipeline
from DORA_XGB import DORA_XGB
import warnings
warnings.simplefilter('ignore')

In [3]:
dataDir = '/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/'
modelBuildingDataDir = os.path.join(dataDir, 'modelBuildingData/')
resultsDir = os.path.join(dataDir, 'Results/')

In [4]:
EnamineAntiviralsData_top20 = pd.read_csv(resultsDir + "virus/EnamineAntiviralsCompounds_allVirus.csv")
EnamineAntiviralsData_top20

,Rank,SMILES,pPotency_prediction,IC50 (M)
0,1,COCCN1C(=NC2=C1C(=O)NC(=O)N2C)N(C)CC=3C=CC=CC3,7.089,8.149146e-08
1,2,COCCN1C(=O)NC(=O)C(=C1N)N(CC=2C=CC=CC2)C(C)=O,6.840,1.445621e-07
2,3,COCCN1C=C(C=CC1=O)NC(=O)C2C(C=C(C)C)C2(C)C,6.794,1.608349e-07
3,4,COCCN1C(=NC2=C1C(=O)NC(=O)N2C)NN=CC=3C=CC=CC3,6.650,2.238847e-07
4,5,COC=1C=CC(OC)=C(C1)C(O)CN2C(C)=NC=3C=CC=CC32,6.643,2.273610e-07
5,6,CCC=1C=C(CC)N(N1)C2=NC3=C(C(=O)NC(=O)N3C)N2CCOC,6.629,2.347164e-07
6,7,CCOCCN1C(=NC2=C1C(=O)NC(=O)N2C)N3N=C(C)C=C3C,6.600,2.510905e-07
7,8,CC=1C=CC=C(N1)NC(=O)NC2CCCN(CC(F)(F)F)C2=O,6.573,2.674945e-07
8,9,CC1=CC=2N=CN(CC(O)CN3C(=O)NC(C)(C)C3=O)C2C=C1C,6.547,2.837417e-07
9,10,COC=1C=CC(=CN1)NC(=O)NCCC(=O)N2CC(C)OC(C)C2,6.529,2.959537e-07


In [5]:
EnamineAntiviralsData_top20_SMILESonly = EnamineAntiviralsData_top20['SMILES']
EnamineAntiviralsData_top20_SMILESonly.to_csv(os.path.join(resultsDir + "virus/EnamineAntiviralsData_top20_SMILESonly.csv"), index=False)
EnamineAntiviralsData_top20_SMILESonly

0      COCCN1C(=NC2=C1C(=O)NC(=O)N2C)N(C)CC=3C=CC=CC3
1       COCCN1C(=O)NC(=O)C(=C1N)N(CC=2C=CC=CC2)C(C)=O
2          COCCN1C=C(C=CC1=O)NC(=O)C2C(C=C(C)C)C2(C)C
3       COCCN1C(=NC2=C1C(=O)NC(=O)N2C)NN=CC=3C=CC=CC3
4        COC=1C=CC(OC)=C(C1)C(O)CN2C(C)=NC=3C=CC=CC32
5     CCC=1C=C(CC)N(N1)C2=NC3=C(C(=O)NC(=O)N3C)N2CCOC
6        CCOCCN1C(=NC2=C1C(=O)NC(=O)N2C)N3N=C(C)C=C3C
7          CC=1C=CC=C(N1)NC(=O)NC2CCCN(CC(F)(F)F)C2=O
8      CC1=CC=2N=CN(CC(O)CN3C(=O)NC(C)(C)C3=O)C2C=C1C
9         COC=1C=CC(=CN1)NC(=O)NCCC(=O)N2CC(C)OC(C)C2
10       CC(NC(=O)NC1=CC=CN=C1N2CCCCC2)C(=O)NC(C)(C)C
11          COCCN1C(=NC=2C=CC=CC21)C=3C=CC(OC)=C(O)C3
12      COCCN1C(=NC=2C=CC=CC21)C=3C=C(OC)C(O)=C(C3)OC
13                    COC=1N=CC=CC1NC(=O)NCCN2CCCCC2C
14                 COC=1N=CC=CC1NC(=O)NCCC(=O)N2CCCC2
15              COC=1C=CC(=CN1)NC(=O)NCCC(=O)N2CCOCC2
16         CC(C)=CC1C(C(=O)NC=2C=CN(CC(N)=O)N2)C1(C)C
17           COCCOC=1C=CC(=CN1)NC(=O)C=2CCC(=O)N(C)N2
18        COCCN1C=C(C=C(C#N)

Define parameters for BioPKS

In [19]:
pathway_sequence = ['pks','bio']  # choose between ['pks'] or ['pks','bio']
target_smiles = 'COCCN1C(=NC2=C1C(=O)NC(=O)N2C)N(C)CC=3C=CC=CC3'
target_name = 'gluconic_lactone'
pks_release_mechanism = 'thiolysis' # choose from 'cyclization' or 'thiolysis'

Define configuration file path for BioPKS

In [20]:
config_filepath = os.path.join('/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/PubchemData/BioPKS_config_file.json')

we initialize DORA-XGB, a supervised learning classifier that can help predict the feasibility of reactions catalyzed by regular, monofunctional enzymes in Biology.

In [21]:
post_pks_rxn_model = DORA_XGB.feasibility_classifier(cofactor_positioning = 'add_concat',
                                                     model_type = "spare")

initialize an object of the biopks_pipeline class

In [22]:
biopks_pipeline_object = biopks_pipeline.biopks_pipeline(
                                             pathway_sequence = pathway_sequence,
                                             target_smiles = target_smiles,
                                             target_name = target_name,
                                             feasibility_classifier = post_pks_rxn_model,
                                             pks_release_mechanism = pks_release_mechanism,
                                             config_filepath = config_filepath)


Extender units successfully chosen for polyketide synthases

Starter units successfully chosen for polyketide synthases


begin a combined PKS and post-PKS synthesis with BioPKS Pipeline

In [23]:
### ----- Start synthesis -----
if __name__ == "__main__":
    biopks_pipeline_object.run_combined_synthesis(max_designs = 4)
    biopks_pipeline_object.save_results_logs()


Starting PKS synthesis with RetroTide
---------------------------------------------
computing module 1
   testing 120 designs
   best score is 0.14814814814814814
computing module 2
   testing 600 designs
   best score is 0.13793103448275862

Best PKS design: [["AT{'substrate': 'Malonyl-CoA'}", 'loading: True'], ["AT{'substrate': 'Methoxymalonyl-CoA'}", "KR{'type': 'B1'}", 'DH{}', 'loading: False']]

Closest final product is: CC=C(OC)C(=O)O

Finished PKS synthesis: closest product to the target using the top PKS design of [["AT{'substrate': 'Malonyl-CoA'}", 'loading: True'], ["AT{'substrate': 'Methoxymalonyl-CoA'}", "KR{'type': 'B1'}", 'DH{}', 'loading: False']] is CC=C(OC)C(=O)O.

Moving onto non-PKS modifications...
Job name: gluconic_lactone_PKS0_BIO1
Job type: enzymatic network expansion forward
Job started on: 2025-11-12 15:55:40.580144
Number of generations: 1
Number of operators loaded: 3571
Number of molecules before expantion (including cofactors): 42
Number of molecules afte

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/BioPKS-Pipeline/biopks_pipeline//users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/BioPKS/virus//gluconic_lactone_PKS_BIO1_config.json'